In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotnine as gg

import load_rapp
import load_bigg

### Rapp data loading

In [ ]:
# load LFC raw measurements
rapp_metabolomic_df = load_rapp.load_metabolomics_matrix()
assert rapp_metabolomic_df.index.is_unique
rapp_metabolomic_df["base_id"] = load_bigg.rapp_to_bigg_index(
    rapp_metabolomic_df["Abbrev"]
)

# load annotations for metabolites that are annotated in BiGG
bigg_metabolite_ref = load_rapp.load_metabolite_reference().set_index("Abbreviation")


# Large metabolites appear as both [M+H]+ and [M+3H]3+ adducts → duplicate base_ids.
# Keep the adduct with the most non-NaN observations.
sample_cols = [
    c
    for c in rapp_metabolomic_df.columns
    if c not in ["base_id", "Abbrev", "Metabolite", "Mass", "KEGG"]
]
metabolite_info = (
    rapp_metabolomic_df[rapp_metabolomic_df["base_id"].isin(bigg_metabolite_ref.index)]
    .assign(n_obs=lambda df: df[sample_cols].notna().sum(axis=1))
    .sort_values("n_obs")
    .loc[lambda x: x["n_obs"] > 0]  # remove always NaN metabolites
    .loc[lambda df: ~df["base_id"].duplicated(keep="last")][
        ["base_id", "Abbrev", "Metabolite", "Mass", "KEGG", "n_obs"]
    ]
)
print(f"{len(metabolite_info)} metabolites mapped to BiGG (unique base_id)")
metabolite_info

In [ ]:
rapp_metabolite_measurements = (
    rapp_metabolomic_df.loc[metabolite_info.index]
    .set_index("base_id")
    .drop(columns=["Abbrev", "Metabolite", "Mass", "KEGG"])
    .T
)

rapp_sample_info = load_rapp.load_endpoint_od().set_index("Sample_ID")
rapp_sample_info

In [ ]:
rapp_lfc = np.log(rapp_metabolite_measurements)
unique_genes = rapp_sample_info["Gene"].unique()

rapp_avg_lfc_by_gene = []
for gene in unique_genes:
    samples = rapp_sample_info.loc[lambda df: df["Gene"] == gene].index
    avg_lfc = rapp_lfc.loc[samples].mean(0)
    rapp_avg_lfc_by_gene.append(avg_lfc.rename(gene))
rapp_avg_lfc_by_gene = pd.DataFrame(
    rapp_avg_lfc_by_gene,
    index=unique_genes,
    columns=rapp_metabolite_measurements.columns,
)
rapp_avg_lfc_by_gene.to_csv("rapp_avg_lfc_by_gene.csv")

### Bigg Metabolic model

In [ ]:
bigg_metabolite_metadata = load_bigg.bigg_metabolite_df().reset_index()
bigg_reaction_df = load_bigg.bigg_reaction_df()
bigg_stoichiometric_df = load_bigg.stoichiometry_long()
bigg_metabolite_metadata

In [ ]:
bigg_base_metadata = bigg_metabolite_metadata.groupby("base_id")["id"].unique()
bigg_base_metadata = bigg_base_metadata.to_frame(name="unique_metabolites")
bigg_base_metadata["n_unique_metabolites"] = bigg_base_metadata[
    "unique_metabolites"
].str.len()
bigg_base_metadata

In [ ]:
bigg_metabolite_metadata["compartment"].unique()

In [ ]:
bigg_stoichiometric_df.groupby("base_id")["metabolite_id"]

In [ ]:
bigg_stoichiometric_df.loc[lambda x: x["base_id"] == "12dgr141"].sort_values(
    "metabolite_id"
)

In [ ]:
bigg_stoichiometric_df.groupby("base_id")["metabolite_id"].nunique().value_counts()

In [ ]:
bigg_stoichiometric_df

### intersection

In [ ]:
metabolite_info

In [ ]:
bigg_base_metadata

In [ ]:
metabolite_info

In [ ]:
metabolite_info.merge(bigg_base_metadata, left_on="base_id", right_index=True)[
    "n_unique_metabolites"
].value_counts()

In [ ]:
bigg_stoichiometric_df

In [ ]:
metabolite_info

In [ ]:
covered_reactions = metabolite_info.merge(
    bigg_stoichiometric_df, left_on="base_id", right_on="base_id"
)
covered_reactions["reaction_id"].unique().shape

In [ ]:
bigg_stoichiometric_df["reaction_id"].unique().shape

### Clustering of all observations

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

# rapp_metabolite_measurements_ = (
#     rapp_metabolite_measurements - rapp_metabolite_measurements.mean(0)
# )
# rapp_metabolite_measurements_ = (
#     rapp_metabolite_measurements_ / rapp_metabolite_measurements_.std(0)
# )
rapp_metabolite_measurements_ = rapp_metabolite_measurements.copy()
pca = PCA()
rapp_metabolite_pcs = pca.fit_transform(rapp_metabolite_measurements_)


plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.show()

In [ ]:
import scanpy as sc

adata_mock = sc.AnnData(X=rapp_metabolite_pcs[:, :50])
sc.pp.neighbors(adata_mock, n_neighbors=10, use_rep="X")

sc.tl.umap(adata_mock)
sc.tl.leiden(adata_mock, resolution=1.0, flavor="igraph")
rapp_sample_info["cluster_assignment"] = adata_mock.obs["leiden"].values

rapp_sample_info["UMAP1"] = adata_mock.obsm["X_umap"][:, 0]
rapp_sample_info["UMAP2"] = adata_mock.obsm["X_umap"][:, 1]

In [ ]:
for cluster in sorted(rapp_sample_info["cluster_assignment"].unique()):
    print(f"Cluster {cluster}:")
    print(
        rapp_sample_info.loc[rapp_sample_info["cluster_assignment"] == cluster][
            "Gene"
        ].unique()
    )

I am clustering CRISPRi gene knockouts based on their metabolomic profiles (measurements of enrichment/depletion of various metabolites).
Each gene knockdown has two replicates.

Organism: E. coli
Medium: LB

 1. Assess if beyond reproducibility variations there is relevant signal
 2. if so, annotate each cluster

In [ ]:
(
    gg.ggplot(
        rapp_sample_info.reset_index(),
        gg.aes(x="UMAP1", y="UMAP2", color="cluster_assignment"),
    )
    + gg.geom_point()
    + gg.theme(legend_position="none")
)

In [ ]:
(
    gg.ggplot(rapp_sample_info.reset_index(), gg.aes(x="UMAP1", y="UMAP2"))
    + gg.geom_point()
    + gg.geom_point(
        rapp_sample_info.loc[
            lambda x: x["Gene"].isin(["lpxA", "lpxC", "lpxD", "lpxB", "lpxK"])
        ].reset_index(),
        gg.aes(x="UMAP1", y="UMAP2", color="Gene"),
    )
)

In [ ]:
(
    gg.ggplot(
        rapp_sample_info.reset_index(), gg.aes(x="UMAP1", y="UMAP2", color="Well")
    )
    + gg.geom_point()
    + gg.theme(legend_position="none")
)

In [ ]:
(
    gg.ggplot(
        rapp_sample_info.reset_index(), gg.aes(x="UMAP1", y="UMAP2", color="Replicate")
    )
    + gg.geom_point()
    + gg.theme(legend_position="none")
)

In [ ]:
import plotly.express as px

px.scatter(
    rapp_sample_info.reset_index(),
    x="UMAP1",
    y="UMAP2",
    color="Gene",
    hover_data=["Gene"],
)

### Replicate variability

In [ ]:
rapp_sample_info["Gene"].unique()

In [ ]:
from sklearn.decomposition import PCA

gene_samples = rapp_sample_info.loc[rapp_sample_info["Gene"] == "aas"].index
gene_measurements = rapp_metabolite_measurements.loc[gene_samples]

plt.scatter(gene_measurements.iloc[0], gene_measurements.iloc[1])

### Clustering based on replicate averages

In [ ]:
rapp_metabolite_measurements

In [ ]:
from sklearn.decomposition import PCA

rapp_metabolite_measurements_ = []
genes = rapp_sample_info["Gene"].unique()
for gene in sorted(genes):
    gene_samples = rapp_sample_info.loc[rapp_sample_info["Gene"] == gene].index
    gene_measurements = rapp_metabolite_measurements.loc[gene_samples]
    rapp_metabolite_measurements_.append(gene_measurements.mean(0).values)
rapp_metabolite_measurements_ = np.array(rapp_metabolite_measurements_)

In [ ]:
rapp_metabolite_measurements_.shape

In [ ]:
pca = PCA()
rapp_metabolite_pcs = pca.fit_transform(rapp_metabolite_measurements_)


plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.show()

In [ ]:
rapp_metabolite_pcs.shape

In [ ]:
import scanpy as sc

adata_mock = sc.AnnData(X=rapp_metabolite_pcs[:, :50])
sc.pp.neighbors(adata_mock, n_neighbors=10, use_rep="X")

sc.tl.umap(adata_mock)
sc.tl.leiden(adata_mock, resolution=0.5, flavor="igraph")

plot_df = pd.DataFrame(
    {
        "UMAP1": adata_mock.obsm["X_umap"][:, 0],
        "UMAP2": adata_mock.obsm["X_umap"][:, 1],
        "cluster_assignment": adata_mock.obs["leiden"].values,
        "Gene": genes,
    }
)

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="UMAP1", y="UMAP2", color="cluster_assignment"))
    + gg.geom_point()
    + gg.theme()
    + gg.scale_color_brewer(type="qual", palette="Set3")
)

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="UMAP1", y="UMAP2"))
    + gg.geom_point()
    + gg.geom_point(
        plot_df.loc[lambda x: x["Gene"].isin(["lpxA", "lpxC", "lpxD", "lpxB", "lpxK"])],
        gg.aes(x="UMAP1", y="UMAP2", color="Gene"),
        size=3,
    )
)

In [ ]:
for cluster in sorted(plot_df["cluster_assignment"].unique()):
    print(f"Cluster {cluster}:")
    print(plot_df.loc[plot_df["cluster_assignment"] == cluster]["Gene"].unique())

### BiGG data loading

In [ ]:
bigg_metabolite_metadata = load_bigg.bigg_metabolite_df().set_index("base_id")
bigg_reaction_df = load_bigg.bigg_reaction_df()
bigg_stoichiometric_df = load_bigg.stoichiometry_long()
bigg_metabolite_metadata